# score_runs_qlora_q309 — GPT-4.1 OmniContext scoring of the NF4 quantization probe
Identical to `score_runs`: same `evaluate()` break patch, same resize convention, same resumable ThreadPool scorer with the uid guard and the guessed-score / unparsable-score filters. Only the config cell changes — `DATASET=test`, `RUN=qlora_nf4_q309_n24`, `BASE_RUN=baseline`.

Scores the 24-row stratified sanity set generated by `qlora_quant_probe_q309`, in which all 309 linear modules were quantized to NF4. The row list is derived from the PNGs present in the run directory, so only those 24 are scored.

Section 6 pairs against the bf16 baseline on the same 24 rows. Two cautions on reading it. The vs-baseline delta mixes quantization damage with ordinary sample-to-sample variation, whose measured floor is sigma_d = 0.966, giving a standard error near 0.20 at n=24. The decision this probe exists for is the q309-vs-q302 comparison, which is clean: both runs are NF4 and differ only in seven modules.

## 1. Repo, scorer patch

In [1]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [2]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET  = "test"        # the 488-row held-out evaluation set
QUANT    = "q309"        # q309: all linear modules NF4 | q302: seven bottleneck modules kept bf16
RUN      = f"qlora_nf4_{QUANT}_n24"
BASE_RUN = "baseline"    # bf16 baseline, paired on the 24 scored rows

GEN    = f"{DRIVE}/gen/{DATASET}/{RUN}"
SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}.jsonl"
os.makedirs(os.path.dirname(SCORES), exist_ok=True)

have = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())
assert have, f"no generated images in {GEN}"
print(f"images : {GEN} -> {len(have)} (rows {have[0]}..{have[-1]})")
print(f"scores : {SCORES}")
man = f"{GEN}/manifest.json"
m = json.load(open(man)) if os.path.exists(man) else {}
if m:
    print(f"manifest: {m.get('gpu')} | torch {m.get('torch')} | written {m.get('written')}")
else:
    print("manifest: absent")

# qlora_quant_probe writes no manifest. Its generation settings are fixed in the notebook:
# 512x512, align_res=False, 50 steps, text guidance 5.0, image guidance 2.0, cfg_range (0,1),
# max_sequence_length 1024, seed 0, shipped negative prompt.
print("config  : qlora_quant_probe, seed 0, 50 steps, tg 5.0 / ig 2.0, shipped negative prompt")

Mounted at /content/drive
images : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/test/qlora_nf4_q309_n24 -> 24 (rows 8..480)
scores : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/test__qlora_nf4_q309_n24.jsonl
manifest: absent
config  : qlora_quant_probe, seed 0, 50 steps, tg 5.0 / ig 2.0, shipped negative prompt


## 3. Reference images

In [3]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test", "screen", "screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
assert max(have) < len(rows), f"image index {max(have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | scoring {len(have)}")

restored 37063 / 37063
test: 488 rows | scoring 24


## 4. Sanity — 3 rows

In [4]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

for i in have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

row 8: PF=8  SC=6
   PF: The final image depicts a young woman with long brown wavy hair, smiling brightly at the camera while holding a smartphone, poised to take a
   SC: The final image (second) shows a young woman smiling in a white shirt, holding a smartphone in selfie mode, which closely matches the prompt

row 13: PF=4  SC=9
   PF: The final image (image 4) shows a young woman with long brown wavy hair, high cheekbones, and a broad bright smile. She is wearing a crisp w
   SC: The final image (image 4) depicts a woman whose facial features – including high cheekbones, bright smile, skin tone, and long wavy brown ha

row 44: PF=9  SC=9
   PF: The final image accurately depicts two businessmen in a modern office environment, correctly reflecting the instructed roles: one is intentl
   SC: The final image displays two businessmen at a desk, matching the office scenario described. The man reviewing documents is consistent with t



## 5. Full pass (resumable)

In [5]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

done = set()
if os.path.exists(SCORES):
    for l in open(SCORES, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])   # failures stay retryable
        except: pass

def score_row(i):
    r  = rows[i]
    gp = f"{GEN}/{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
    # evaluate() substitutes random.randint(0,10) after 3 failed parses; that is not a measurement
    if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
        return {"i": i, "error": "guessed_score"}
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
            "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
            "PF": pf, "SC": sc,
            "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

todo = [i for i in have if i not in done]
print(f"to score: {len(todo)} (done: {len(done)})")
with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_row, i) for i in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 25 == 0: print(f"  {n}/{len(todo)}")
print("scoring done ->", SCORES)

to score: 0 (done: 24)
scoring done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/test__qlora_nf4_q309_n24.jsonl


## 6. Aggregate, paired delta vs BASE_RUN

In [6]:
import statistics as st
from collections import defaultdict

SIGMA_ITEM = 0.362      # judge re-scoring SD on an identical image; RMS over sd_reps (90 rows x5, ddof=1)

def load(run):
    d, errs = {}, defaultdict(int)
    for l in open(f"{DRIVE}/scores/{DATASET}__{run}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r["i"]] = r
    return d, errs

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
cur, errs = load(RUN)
print(f"{DATASET}/{RUN}: {len(cur)} scored" + (f" | errors {dict(errs)}" if errs else ""))
print(f"  PF {st.mean([d['PF'] for d in cur.values()]):.3f}  "
      f"SC {st.mean([d['SC'] for d in cur.values()]):.3f}  "
      f"Overall {st.mean([ov(d) for d in cur.values()]):.3f}")
for key in ["n_refs", "ref_kind"]:
    g = defaultdict(list)
    for d in cur.values(): g[d[key]].append(ov(d))
    print("  by " + key + ": " + " | ".join(f"{k}: {st.mean(v):.3f} (n={len(v)})" for k, v in sorted(g.items(), key=str)))

if RUN != BASE_RUN:
    from scipy.stats import ttest_rel
    base, _ = load(BASE_RUN)
    I = sorted(set(cur) & set(base))
    # a row index means nothing on its own; the uid is what identifies the data row
    bad = [i for i in I if cur[i]["uid"] != base[i]["uid"]]
    assert not bad, f"uid mismatch on rows {bad[:5]} — runs are not aligned"
    print(f"\npaired delta vs {BASE_RUN} (n={len(I)})")
    for lab, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
        d = [fn(cur[i]) - fn(base[i]) for i in I]
        t, p = ttest_rel([fn(cur[i]) for i in I], [fn(base[i]) for i in I])
        print(f"  {lab:8s} mean {st.mean(d):+.3f}  SD {st.stdev(d):.3f}  "
              f"SE {st.stdev(d)/math.sqrt(len(d)):.3f}  t {t:+.2f}  p {p:.3f}")

    sd    = st.stdev([ov(cur[i]) - ov(base[i]) for i in I])
    floor = math.sqrt(2) * SIGMA_ITEM
    excess = math.sqrt(max(sd*sd - floor*floor, 0))
    print(f"\n  sigma_d (Overall)   = {sd:.3f}")
    print(f"  judge-only floor    = {floor:.3f}   (sqrt(2) * {SIGMA_ITEM})")
    print(f"  excess over floor   = {excess:.3f}")
    print(f"  -> implied per-image sigma if this pair differs only by generation noise: {excess/math.sqrt(2):.3f}")
    print("     (valid only when RUN re-runs BASE_RUN's config; otherwise excess = generation + prompt effect)")
    for delta in [0.1, 0.2, 0.3]:
        print(f"  rows for 80% power at true delta {delta}: {math.ceil(7.85*sd*sd/(delta*delta))}")
else:
    print("\nbaseline run: no comparator")

test/qlora_nf4_q309_n24: 24 scored
  PF 7.792  SC 8.375  Overall 8.003
  by n_refs: 1: 8.576 (n=6) | 2: 7.965 (n=9) | 3: 7.660 (n=9)
  by ref_kind: all_up: 7.955 (n=9) | mixed: 7.623 (n=6) | no_up: 8.305 (n=9)

paired delta vs baseline (n=24)
  PF       mean -0.083  SD 1.248  SE 0.255  t -0.33  p 0.747
  SC       mean +0.042  SD 0.955  SE 0.195  t +0.21  p 0.833
  Overall  mean -0.074  SD 0.896  SE 0.183  t -0.41  p 0.688

  sigma_d (Overall)   = 0.896
  judge-only floor    = 0.512   (sqrt(2) * 0.362)
  excess over floor   = 0.736
  -> implied per-image sigma if this pair differs only by generation noise: 0.520
     (valid only when RUN re-runs BASE_RUN's config; otherwise excess = generation + prompt effect)
  rows for 80% power at true delta 0.1: 631
  rows for 80% power at true delta 0.2: 158
  rows for 80% power at true delta 0.3: 71
